# CUAD - Veri Kesfi ve RAG Prototipi

**Eksun Holding - AI Engineer Vaka Calismasi - Proje B: Yerel LLM ile Hukuki Metin Analizi**

Bu notebook deneysel calisma alani. Buradaki kod olgunlastikca `rag/` paketindeki modullere tasinacak.

## 0. Kurulum kontrolu

In [ ]:
import sys
from pathlib import Path

# Proje kokunu path'e ekle (notebook, notebooks/ altinda calisiyor)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Proje kok dizini: {PROJECT_ROOT}")

In [ ]:
from rag.config import settings

print(settings)

## 1. Veri Kesfi (EDA)

CUAD_v1.zip'i `data/raw/` altina cikardigindan emin ol (bkz. README.md - Kurulum).

Beklenen dosyalar:
- `data/raw/full_contract_txt/*.txt`
- `data/raw/CUAD_v1.json`
- `data/raw/master_clauses.csv`

In [ ]:
from pathlib import Path

raw_dir = PROJECT_ROOT / "data" / "raw"
txt_files = sorted((raw_dir / "full_contract_txt").glob("*.txt")) if (raw_dir / "full_contract_txt").exists() else []
print(f"Bulunan sozlesme sayisi: {len(txt_files)}")
if txt_files:
    print(txt_files[0].name)
    print(txt_files[0].read_text(encoding='utf-8', errors='ignore')[:1000])

In [ ]:
import pandas as pd

master_csv = raw_dir / "master_clauses.csv"
if master_csv.exists():
    df = pd.read_csv(master_csv)
    print(df.shape)
    df.head()

## 2. Chunking Denemesi

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    separators=["\n\n", "\n", ". ", " "],
)

if txt_files:
    sample_text = txt_files[0].read_text(encoding='utf-8', errors='ignore')
    sample_chunks = splitter.split_text(sample_text)
    print(f"Ornek sozlesme {len(sample_chunks)} chunk'a bolundu")
    print('---')
    print(sample_chunks[0][:500])

## 3. Embedding Denemesi (local, HuggingFace sentence-transformers)

In [ ]:
from rag.embeddings import get_embedding_model

embedder = get_embedding_model()
if txt_files:
    vec = embedder.embed_query(sample_chunks[0])
    print(f"Embedding boyutu: {len(vec)}")

## 4. Vektor Store Denemesi (ChromaDB)

In [ ]:
# from db.vectorstore import get_vectorstore, reset_collection
# reset_collection()
# vs = get_vectorstore()
# TODO: chunk'lari add et, similarity_search dene

## 5. Local LLM ile Uctan Uca Deneme (Ollama)

Onceden terminalde calistir:
```bash
ollama pull llama3.1:8b
ollama serve
```

In [ ]:
# from rag.pipeline import get_llm
# llm = get_llm()
# response = llm.invoke("Bu bir baglanti testidir. Kisaca 'merhaba' de.")
# print(response.content)

## 6. Sonraki Adimlar

- [ ] `rag/ingest.py` -> `load_contracts()` implement et
- [ ] `rag/chunking.py` -> `chunk_documents()` implement et
- [ ] `db/vectorstore.py` -> `add_chunks()` implement et
- [ ] `rag/retriever.py` -> `retrieve()` implement et
- [ ] `rag/pipeline.py` -> `answer_question()` implement et
- [ ] Streamlit app'i `rag.pipeline.answer_question` ile bagla
- [ ] CUAD_v1.json'daki gercek soru-cevaplarla hizli bir dogruluk testi yap